### Summarize parallel validation results

**Purpose:** Combine customer and order checks after both branches finish.

**Inputs:** Task values emitted by `customer_check` and `order_check`.

**Outputs:** A consolidated PASS/FAIL summary for the final Job gate.

**Why it matters:** The join point proves that both independent checks completed before the workflow advances.

In [0]:
# Purpose: collect results from both parallel branches.

customer_rows = dbutils.jobs.taskValues.get(
    taskKey="customer_check",
    key="customer_rows",
    debugValue=99441
)

customer_status = dbutils.jobs.taskValues.get(
    taskKey="customer_check",
    key="customer_status",
    debugValue="PASS"
)

order_item_rows = dbutils.jobs.taskValues.get(
    taskKey="order_check",
    key="order_item_rows",
    debugValue=112650
)

distinct_orders = dbutils.jobs.taskValues.get(
    taskKey="order_check",
    key="distinct_orders",
    debugValue=98666
)

order_status = dbutils.jobs.taskValues.get(
    taskKey="order_check",
    key="order_status",
    debugValue="PASS"
)

overall_status = (
    "PASS"
    if customer_status == "PASS" and order_status == "PASS"
    else "FAIL"
)

summary_df = spark.createDataFrame(
    [
        (
            customer_rows,
            order_item_rows,
            distinct_orders,
            customer_status,
            order_status,
            overall_status
        )
    ],
    [
        "customer_rows",
        "order_item_rows",
        "distinct_orders",
        "customer_status",
        "order_status",
        "overall_status"
    ]
)

display(summary_df)

if overall_status != "PASS":
    raise RuntimeError("Parallel workflow validation failed")

print("SUCCESS: parallel learning workflow passed")